# Lab 01: First agent from scratch

Build a working ReAct-style agent in pure Python — no framework. By the end of
this notebook you'll have an agent that uses tools, handles errors, and stops
when it should.

This notebook is the runnable companion to
[`labs/01-first-agent-from-scratch/README.md`](./README.md). Read the brief first
if you haven't.

**Estimated time:** 60–90 minutes.
**Difficulty:** 🟢 Beginner.
**Prerequisites:**
[`concepts/agents/what-is-an-agent.md`](../../concepts/agents/what-is-an-agent.md),
[`concepts/agents/agent-loop.md`](../../concepts/agents/agent-loop.md),
[`concepts/agents/react-pattern.md`](../../concepts/agents/react-pattern.md).

## Step 0: Setup

We're going to use OpenAI as the default provider. If you'd rather use
Anthropic, set `PROVIDER = "anthropic"` two cells down — the rest of the
notebook is provider-agnostic.

Make sure you've followed [`setup/`](../../setup/) and your `.env` file has
at least one of `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` set.

In [ ]:
# Load environment variables from .env in the repo root
import os
import pathlib

from dotenv import load_dotenv

# Walk up to find the repo root (the dir containing .env.example)
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

# Confirm at least one key is available
has_openai = bool(os.getenv("OPENAI_API_KEY"))
has_anthropic = bool(os.getenv("ANTHROPIC_API_KEY"))
print(f"OpenAI key present:    {has_openai}")
print(f"Anthropic key present: {has_anthropic}")
assert has_openai or has_anthropic, "Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env"


**Sample output:**

```
OpenAI key present:    True
Anthropic key present: False
```

(One True is enough. Both is fine.)

In [ ]:
# Pick your provider
PROVIDER = "openai"     # or "anthropic"

# Pick the model. Smaller/cheaper models are fine for this lab.
MODEL = {
    "openai": "gpt-4o-mini",
    "anthropic": "claude-haiku-4-5-20251001",
}[PROVIDER]

print(f"Using {PROVIDER} with model {MODEL}")


## Step 1: The bare minimum

Before we build an agent, let's confirm the *problem*. We'll ask an LLM a
question that requires multi-step arithmetic. Models are notoriously bad at
this without tools — but you should verify it yourself, on your provider, with
your specific model. Don't trust the lab; trust the run.

In [ ]:
# A minimal provider-agnostic chat wrapper.
# We'll grow this as we go. For now it just produces text.

from typing import Any

def chat(messages: list[dict], **kwargs: Any) -> str:
    """Send messages to the configured provider and return the text response."""
    if PROVIDER == "openai":
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            **kwargs,
        )
        return resp.choices[0].message.content or ""
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        # Anthropic separates system messages from the conversation
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        resp = client.messages.create(
            model=MODEL,
            system=system,
            messages=non_system,
            max_tokens=1024,
            **kwargs,
        )
        return resp.content[0].text
    else:
        raise ValueError(f"Unknown provider: {PROVIDER}")

# Try it.
question = (
    "What's 17% of the average of 234, 891, and 1502? "
    "And is that result greater than 100?"
)
answer = chat([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": question},
])
print(answer)


**Sample output (will vary):**

```
The average of 234, 891, and 1502 is (234 + 891 + 1502) / 3 = 2627 / 3 ≈ 875.67.
17% of 875.67 is approximately 148.86.
Yes, 148.86 is greater than 100.
```

Often correct, sometimes not. Even when it's right, you have no way to *verify*
without doing the math yourself. The model arrived at the answer by predicting
plausible-looking arithmetic — it didn't actually compute anything. The whole
point of giving it a calculator is to ground the computation in a real
operation.

## Step 2: Define tools

Two tools to start: a calculator and a mock web search. Both use Pydantic for
typed arguments and structured returns. The schema we expose to the LLM is
generated from these models — no hand-written JSON Schema, no string parsing.

In [ ]:
from collections.abc import Callable

from pydantic import BaseModel, Field

# ─── Tool 1: Calculator ───────────────────────────────────────────────────────

class CalculatorArgs(BaseModel):
    expression: str = Field(
        description=(
            "A Python arithmetic expression to evaluate. "
            "Only numbers, +, -, *, /, %, parentheses, and decimal points. "
            "Example: '(234 + 891 + 1502) / 3'"
        )
    )

def calculator(args: CalculatorArgs) -> dict:
    """Evaluate a simple arithmetic expression and return the numeric result."""
    expr = args.expression
    # Whitelist characters to avoid eval surprises. This is a teaching example,
    # not a sandbox — in production, use a proper expression parser.
    allowed = set("0123456789+-*/().%  ")
    if not set(expr) <= allowed:
        raise ValueError(
            f"Expression contains disallowed characters: {set(expr) - allowed}"
        )
    result = eval(expr, {"__builtins__": {}}, {})  # nosec — whitelisted
    return {"result": result}


# ─── Tool 2: Web search (mocked) ──────────────────────────────────────────────

class WebSearchArgs(BaseModel):
    query: str = Field(description="The web search query.")

def web_search(args: WebSearchArgs) -> dict:
    """Mock web search. Returns canned results so the lab doesn't need a real API key.

    In a real lab, this would call Tavily, Brave, or another search API. The
    point here is to show the agent integrating *another* tool — not to test
    search quality.
    """
    query = args.query.lower()
    canned = {
        "population of canada": "Canada's population is approximately 41 million (2024 estimate).",
        "population of france": "France's population is approximately 68 million (2024 estimate).",
        "population of japan": "Japan's population is approximately 125 million (2024 estimate).",
    }
    for key, value in canned.items():
        if key in query:
            return {"results": [value]}
    return {"results": [f"No relevant results found for query: {args.query!r}"]}


# A small registry — each tool is (function, args_model)
TOOLS: dict[str, tuple[Callable, type[BaseModel]]] = {
    "calculator": (calculator, CalculatorArgs),
    "web_search": (web_search, WebSearchArgs),
}

print("Defined tools:", list(TOOLS))


Try the calculator directly — make sure it works before involving the LLM.

In [ ]:
# Sanity-check the calculator
result = calculator(CalculatorArgs(expression="(234 + 891 + 1502) / 3"))
print(result)


**Sample output:**

```
{'result': 875.6666666666666}
```

## Step 3: Wire up function calling

The LLM provider needs to know what tools exist. Both OpenAI and Anthropic
accept a list of tool specifications in their API. We'll generate those specs
from the Pydantic models.

In [ ]:
def tool_schemas() -> list[dict]:
    """Build the tool schemas the LLM expects, from the Pydantic models."""
    schemas = []
    for name, (fn, args_model) in TOOLS.items():
        schemas.append({
            "type": "function",
            "function": {
                "name": name,
                "description": (fn.__doc__ or "").strip().split("\n")[0],
                "parameters": args_model.model_json_schema(),
            },
        })
    return schemas


# Inspect what we'll send to the LLM
import json
print(json.dumps(tool_schemas(), indent=2))


**Sample output (abridged):**

```json
[
  {
    "type": "function",
    "function": {
      "name": "calculator",
      "description": "Evaluate a simple arithmetic expression and return the numeric result.",
      "parameters": {
        "properties": {
          "expression": {
            "description": "A Python arithmetic expression to evaluate. ...",
            "type": "string"
          }
        },
        "required": ["expression"],
        "title": "CalculatorArgs",
        "type": "object"
      }
    }
  },
  ...
]
```

Now extend `chat()` to support tool calling and return the raw message instead
of just the text. We'll need the structured tool calls.

In [ ]:
# Provider-agnostic message representation.
# We mimic OpenAI's shape because it's currently the de-facto interchange format;
# providers other than OpenAI translate to/from it.

from dataclasses import dataclass, field

@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict

@dataclass
class AssistantMessage:
    content: str | None
    tool_calls: list[ToolCall] = field(default_factory=list)

def chat_with_tools(messages: list[dict], tools: list[dict] | None = None,
                    tool_choice: str = "auto") -> AssistantMessage:
    """Send messages and return the assistant's structured response."""
    if PROVIDER == "openai":
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice=tool_choice if tools else None,
            temperature=0,
        )
        msg = resp.choices[0].message
        return AssistantMessage(
            content=msg.content,
            tool_calls=[
                ToolCall(id=tc.id, name=tc.function.name,
                         arguments=json.loads(tc.function.arguments))
                for tc in (msg.tool_calls or [])
            ],
        )

    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        # Translate OpenAI-style tool schemas to Anthropic's shape
        anth_tools = [
            {"name": t["function"]["name"],
             "description": t["function"]["description"],
             "input_schema": t["function"]["parameters"]}
            for t in (tools or [])
        ]
        resp = client.messages.create(
            model=MODEL,
            system=system,
            messages=non_system,
            tools=anth_tools or None,
            max_tokens=1024,
        )
        content_text = ""
        tool_calls = []
        for block in resp.content:
            if block.type == "text":
                content_text += block.text
            elif block.type == "tool_use":
                tool_calls.append(ToolCall(
                    id=block.id, name=block.name, arguments=block.input
                ))
        return AssistantMessage(content=content_text or None, tool_calls=tool_calls)
    else:
        raise ValueError(f"Unknown provider: {PROVIDER}")


# Quick sanity check — ask the model to use a tool
msg = chat_with_tools(
    messages=[
        {"role": "system", "content": "You have access to tools. Use them when helpful."},
        {"role": "user", "content": "What is 47 * 83?"},
    ],
    tools=tool_schemas(),
    tool_choice="required",
)
print(f"Content: {msg.content!r}")
print(f"Tool calls: {msg.tool_calls}")


**Sample output (will vary):**

```
Content: None
Tool calls: [ToolCall(id='call_abc123', name='calculator',
                     arguments={'expression': '47 * 83'})]
```

The model emitted a structured tool call. Now we have to execute it.

## Step 4: Build the loop

This is the heart of the agent. We:

1. Sample an action from the LLM ($\pi_\theta(a_t \mid s_t)$).
2. If it's a final answer, return.
3. Otherwise execute each tool call and append the observations to state.
4. Repeat, with a step cap.

The whole thing is one `for` loop.

In [ ]:
MAX_STEPS = 8

def execute_tool(call: ToolCall) -> dict:
    """Dispatch a tool call. Validates args and returns a structured result."""
    if call.name not in TOOLS:
        return {"error": f"Unknown tool: {call.name!r}",
                "available_tools": list(TOOLS)}
    fn, args_model = TOOLS[call.name]
    try:
        # Pydantic validates and coerces types
        args = args_model.model_validate(call.arguments)
        return fn(args)
    except Exception as e:
        # Convert the exception into a structured observation
        return {"error": f"{type(e).__name__}: {e}"}


def run_agent(user_question: str, system_prompt: str | None = None,
              verbose: bool = True) -> str:
    """Run the agent loop until a final answer or step cap."""
    if system_prompt is None:
        system_prompt = (
            "You are a tool-using assistant. Use the available tools when "
            "they would help. When you have enough information to answer, "
            "respond directly without calling another tool."
        )

    state: list[dict] = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question},
    ]

    for step in range(MAX_STEPS):
        if verbose:
            print(f"\n── Step {step + 1} ──")

        msg = chat_with_tools(state, tools=tool_schemas(), tool_choice="auto")

        # Append the assistant's turn to state, preserving tool calls
        state.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ] if msg.tool_calls else None,
        })

        # Terminal: no tool calls means the model is giving a final answer
        if not msg.tool_calls:
            if verbose:
                print(f"Final answer: {msg.content}")
            return msg.content or ""

        # Execute each tool call and append observations
        for call in msg.tool_calls:
            if verbose:
                print(f"  → {call.name}({call.arguments})")
            result = execute_tool(call)
            if verbose:
                print(f"  ← {result}")
            state.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result),
            })

    return "[Agent exhausted step budget without producing a final answer]"


# Run it on our test question
question = (
    "What's 17% of the average of 234, 891, and 1502? "
    "And is that result greater than 100?"
)
result = run_agent(question)
print("\n✓ Lab complete (step 4)")


**Sample output (verbose; details will vary):**

```
── Step 1 ──
  → calculator({'expression': '(234 + 891 + 1502) / 3'})
  ← {'result': 875.6666666666666}

── Step 2 ──
  → calculator({'expression': '875.6666666666666 * 0.17'})
  ← {'result': 148.86333333333332}

── Step 3 ──
Final answer: 17% of the average of 234, 891, and 1502 is approximately
148.86, which is greater than 100.

✓ Lab complete (step 4)
```

A working agent. Three things happened that weren't possible in Step 1:

1. The agent **decomposed** the problem into sub-tasks.
2. Each sub-task was answered by an actual computation, not a guess.
3. The final answer is **grounded** — every number traces to a tool call you can audit.

## Step 5: Add the ReAct thoughts

Modern function-calling APIs let the model emit text content *and* tool calls
in the same response. The text content is the model's thought. Without an
explicit prompt to elicit it, the model often skips the thought and goes
straight to the tool call.

Let's encourage thoughts and see how the runs change.

In [ ]:
REACT_SYSTEM_PROMPT = (
    "You are a tool-using assistant. For every step, follow this pattern:\n"
    "1. Write a brief thought (one or two sentences) about what to do next.\n"
    "2. Call the most appropriate tool, OR give a final answer if you have enough information.\n"
    "Be concise in both thoughts and final answers."
)

# Same question, with the ReAct prompt
result = run_agent(
    "What is 17% of the average of 234, 891, and 1502? "
    "And is the result greater than 100?",
    system_prompt=REACT_SYSTEM_PROMPT,
)


**Sample output (the thoughts appear before the tool calls):**

```
── Step 1 ──
  → calculator({'expression': '(234 + 891 + 1502) / 3'})
  ← {'result': 875.6666666666666}

── Step 2 ──
  → calculator({'expression': '0.17 * 875.6666666666666'})
  ← {'result': 148.86333333333332}

── Step 3 ──
Final answer: I need to compute 17% of the average and compare to 100.
The average of 234, 891, and 1502 is 875.67. 17% of 875.67 is approximately
148.86, which is greater than 100.
```

You'll see varied behavior here depending on the model. Some emit thoughts on
every step; some only on the final response. The behavior to watch for: the
thoughts should *justify* the tool call that follows them. When they don't, the
model is hallucinating its plan.

## Step 6: Handle failures

What happens when a tool raises? Right now, `execute_tool` catches the
exception and returns `{"error": ...}` — so the model sees the error as an
observation and can react.

Let's confirm that path actually works. We'll ask a question that requires the
calculator on a deliberately broken expression first, then see if the agent
recovers.

In [ ]:
# Provoke a controlled failure: ask the model to compute something with
# unfamiliar syntax that won't pass our whitelist.

bad_question = (
    "What is the cosine of pi/4 to four decimal places? "
    "Use the calculator if needed; otherwise reason from first principles."
)

# We don't actually have cos in the calculator, so the model has to either
# fall back to its own reasoning, OR misuse the calculator and recover from
# the error observation. Either is correct.
result = run_agent(bad_question, system_prompt=REACT_SYSTEM_PROMPT)


**Sample output (one of several valid outcomes):**

```
── Step 1 ──
  → calculator({'expression': 'cos(0.7853981633974483)'})
  ← {'error': "ValueError: Expression contains disallowed characters: {'o', 's', 'c'}"}

── Step 2 ──
Final answer: The calculator doesn't support cos. From first principles,
cos(pi/4) = √2/2 ≈ 0.7071.
```

The model recovered: it saw the error observation, recognized the calculator
couldn't handle trig, and answered from memory. That recovery is *only*
possible because the error came back as a structured observation it could read.
A silent crash would have ended the run.

## Step 7: Repeated-action detection

A common failure mode: the agent keeps calling the same tool with the same
arguments because nothing changed. We add a small guard against this.

In [ ]:
def signature(call: ToolCall) -> str:
    """A stable fingerprint of a tool call."""
    return f"{call.name}({json.dumps(call.arguments, sort_keys=True)})"


def run_agent_safe(user_question: str, system_prompt: str | None = None,
                   verbose: bool = True) -> str:
    """Same as run_agent, but bails out on consecutive duplicate tool calls."""
    if system_prompt is None:
        system_prompt = REACT_SYSTEM_PROMPT

    state: list[dict] = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question},
    ]
    last_signature: str | None = None
    duplicate_count = 0

    for step in range(MAX_STEPS):
        if verbose:
            print(f"\n── Step {step + 1} ──")

        msg = chat_with_tools(state, tools=tool_schemas(), tool_choice="auto")
        state.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ] if msg.tool_calls else None,
        })

        if not msg.tool_calls:
            if verbose:
                print(f"Final answer: {msg.content}")
            return msg.content or ""

        for call in msg.tool_calls:
            sig = signature(call)
            if sig == last_signature:
                duplicate_count += 1
                if duplicate_count >= 2:
                    return f"[Halted: agent repeated the same action: {sig}]"
            else:
                duplicate_count = 0
            last_signature = sig

            if verbose:
                print(f"  → {call.name}({call.arguments})")
            result = execute_tool(call)
            if verbose:
                print(f"  ← {result}")
            state.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result),
            })

    return "[Agent exhausted step budget without producing a final answer]"

# Re-run the multi-step arithmetic question to make sure the safe version
# still works the same way on the happy path.
result = run_agent_safe(
    "What's 17% of the average of 234, 891, and 1502? "
    "Is the result greater than 100?",
)


**Sample output:** identical to the Step 5 run. Good — the guard only
kicks in when something goes wrong.

## Step 8: Test scenarios

Let's run a few scenarios that exercise different parts of the agent.

In [ ]:
scenarios = [
    ("Pure arithmetic (calculator)",
     "If a recipe calls for 2.5 cups of flour per serving and I want to make "
     "12 servings, how many cups total? And how many ounces is that, given "
     "8 ounces per cup?"),
    ("Multi-tool (search + calculator)",
     "What is the population of Canada divided by the population of Japan? "
     "Express the answer as a percentage."),
    ("Conversational (no tools needed)",
     "What does the abbreviation 'ReAct' stand for in the context of LLM agents?"),
]

for label, q in scenarios:
    print(f"\n{'═' * 60}\nScenario: {label}\n{'═' * 60}")
    print(f"Q: {q}\n")
    answer = run_agent_safe(q, verbose=False)
    print(f"A: {answer}")


**Sample output (truncated):**

```
════════════════════════════════════════════════════════════
Scenario: Pure arithmetic (calculator)
════════════════════════════════════════════════════════════
Q: If a recipe calls for 2.5 cups of flour per serving and I want
to make 12 servings...

A: You'll need 30 cups of flour, which is equivalent to 240 ounces.

════════════════════════════════════════════════════════════
Scenario: Multi-tool (search + calculator)
════════════════════════════════════════════════════════════
Q: What is the population of Canada divided by the population of Japan?...

A: Canada's population is approximately 41 million and Japan's is
approximately 125 million. The ratio is about 32.8%.

════════════════════════════════════════════════════════════
Scenario: Conversational (no tools needed)
════════════════════════════════════════════════════════════
Q: What does the abbreviation 'ReAct' stand for in the context of LLM agents?

A: ReAct stands for "Reasoning + Acting." It's a prompting pattern from
Yao et al. (ICLR 2023) where the model interleaves natural-language thoughts
with tool calls.
```

The third scenario is important: the agent recognizes when *no* tool is needed
and answers directly. That decision is part of the policy too.

## Step 9 (stretch): Swap the provider

If you have an Anthropic key, change `PROVIDER` and `MODEL` at the top of the
notebook to:

```python
PROVIDER = "anthropic"
MODEL = "claude-haiku-4-5-20251001"   # or another current Anthropic model
```

Restart the kernel and re-run everything. The agent code is unchanged. Only
the `chat_with_tools()` function translates the request to and from
Anthropic's shape.

This is the practical payoff of keeping the agent loop separate from the
provider wrapper. Frameworks do this automatically — but knowing the shape of
the abstraction lets you swap providers in your own code without rewriting the
agent.

## ✓ Lab complete

You've built a working ReAct-style agent in pure Python:

- A provider-agnostic LLM wrapper.
- Pydantic-typed tools with auto-generated schemas.
- The full agent loop with thoughts, actions, observations.
- Structured error handling.
- Repeated-action detection.
- Step-cap enforcement.

Total: about 150 lines of meaningful code, no framework.

### Where to go next

- 🧮 **[`math-foundations/04-agents-as-policies.md`](../../math-foundations/04-agents-as-policies.md)** —
  the math behind the loop you just built.
- 🧮 **[`math-foundations/06-react-formalization.md`](../../math-foundations/06-react-formalization.md)** —
  the math behind the thoughts.
- 🧪 **Lab 02 (forthcoming) — Tool design and selection.** A deeper dive into
  why some tool sets work and others don't.
- 🧪 **Lab 05 (forthcoming) — LangGraph state machine.** The same agent,
  rewritten in LangGraph. See what the framework adds and what stays the same.

### Stretch goals (recap from the README)

- Add a third tool (date/time, unit conversion, or real web search).
- Replace the mock search with [Tavily](https://tavily.com/).
- Add parallel tool calls.
- Add a token-usage tracker.

### Going deeper

🧮 **Math:** The loop you wrote implements
$\pi_\theta(a_t \mid s_t)$ — a policy mapping the conversation state to a
distribution over tool calls and final answers. See
[`math-foundations/04-agents-as-policies.md`](../../math-foundations/04-agents-as-policies.md)
and
[`math-foundations/06-react-formalization.md`](../../math-foundations/06-react-formalization.md).